# Foto → playlist — Parte 2

**Entradas (no Drive):**
- `modelos/foto_humor_energia.joblib`: o modelo da Parte 1 (CLIP + Ridge), com `media_prev` e `desvio_prev`;
- a tabela de notas das músicas, feita pelo MusiCNN/emoMusic no `notas_essentia.ipynb`;
- `audio_teto100.csv`: artista, título, rótulos do MuSe e id do Deezer de cada prévia.

**Etapas:**
1. Conferir as notas das músicas: como se distribuem e se batem com os rótulos de humor.
2. Ver se as fotos caem em regiões que têm música (cobertura).
3. Pôr foto e música na **mesma régua** (z-score) e buscar as músicas mais próximas.

**Sobre a régua:** foto e música usam escalas diferentes (OASIS 1–7, emoMusic 1–9), e as previsões do modelo de foto são comprimidas.
Por isso cada lado é padronizado pela **própria** distribuição: a foto, pela média e desvio das previsões; a música, pela média e desvio das músicas.
Na prática, "uma foto entre as mais calmas" vai receber "músicas entre as mais calmas". Isso é uma **escolha de projeto**, não um fato medido.

## 1. Configuração
Ajuste os caminhos se os seus arquivos tiverem outro nome.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

PASTA = Path("/content/drive/MyDrive/mapa-sonoro")
ARQ_MODELO = PASTA / "modelos" / "foto_humor_energia.joblib"
ARQ_NOTAS_MUSICAS = Path("/content/drive/MyDrive/playlist-foto/notas_musicas.csv")
ARQ_INDICE = None                   # None = procura audio_teto100.csv na pasta do projeto
ARQ_POP = PASTA / "popularidade_final.csv"   # gerado no Mac (buscar_popularidade.py + buscar_fas_artistas.py)
POP_MIN = 0.7                       # mantém só músicas com popularidade >= este percentil (0.5 = metade mais conhecida)
TAMANHO_PLAYLIST = 15
CANDIDATOS_POR_VAGA = 6             # olha as 15×6 = 90 mais próximas
ALFA = 2                            # nota = distância − ALFA × pop (15/11: 0.5 pesava pouco; 2 → 73% das músicas no top 10% de pop)
MAX_POR_ARTISTA = 1                 # variedade: no máximo N músicas do mesmo artista

def achar(nome_ou_padrao):
    achados = sorted(PASTA.rglob(nome_ou_padrao), key=lambda p: -p.stat().st_mtime)
    return achados[0] if achados else None

if ARQ_INDICE is None:
    ARQ_INDICE = achar("audio_teto100.csv")
import re, pandas as pd
PAD_VAL, PAD_ENE = r"valen|valê|^val$|^val_", r"arous|^ener$|^energia$|^energy$"

def tem_notas(p):
    # True se o CSV tem uma coluna de valência E uma de energia/arousal
    try:
        cols = pd.read_csv(p, nrows=0).columns
    except Exception:
        return False
    return any(re.search(PAD_VAL, c, re.I) for c in cols) and any(re.search(PAD_ENE, c, re.I) for c in cols)

if ARQ_NOTAS_MUSICAS is None:       # o CSV mais recente que tenha colunas de valência e energia
    todos_csv = [p for p in sorted(PASTA.rglob("*.csv"), key=lambda p: -p.stat().st_mtime)
                 if p.name not in ("audio_teto100.csv", "OASIS.csv") and "oasis" not in str(p).lower()]
    candidatos = [p for p in todos_csv if tem_notas(p)]
    print("CSVs na pasta:", [str(p.relative_to(PASTA)) for p in todos_csv[:15]])
    print("com valência + energia:", [str(p.relative_to(PASTA)) for p in candidatos])
    ARQ_NOTAS_MUSICAS = candidatos[0] if candidatos else None

print("modelo :", ARQ_MODELO, ARQ_MODELO.exists())
print("notas  :", ARQ_NOTAS_MUSICAS)
print("índice :", ARQ_INDICE)
assert ARQ_NOTAS_MUSICAS is not None, ("Nenhum CSV com colunas de valência e energia foi achado. "
    "Defina ARQ_NOTAS_MUSICAS com o arquivo que o notas_essentia.ipynb gravou")
assert ARQ_INDICE is not None, "Suba data/audio_teto100.csv do Mac para a pasta do projeto no Drive"

## 2. Ler e juntar as tabelas
A ligação entre as duas tabelas é o **`deezer_id`** (se existir nas notas) ou o **nome do arquivo da prévia**
(ex.: `6d26ab8a96aa.mp3` → `6d26ab8a96aa`).
As colunas de nota são detectadas pelo nome. Se a detecção errar, preencha `COL_VAL` e `COL_ENE`.

In [ ]:
import numpy as np

notas = pd.read_csv(ARQ_NOTAS_MUSICAS)
indice = pd.read_csv(ARQ_INDICE)
indice = indice[indice.status == "ok"].copy()
print("arquivo de notas:", ARQ_NOTAS_MUSICAS.name)
print("colunas:", list(notas.columns))
print(notas.head(3).to_string())

COL_VAL = None   # ex.: "valence"
COL_ENE = None   # ex.: "arousal"
COL_ARQ = None   # coluna com o nome/caminho do arquivo (se não houver deezer_id)

def detectar(padrao):
    for c in notas.columns:
        if re.search(padrao, c, re.I):
            return c
COL_VAL = COL_VAL or detectar(PAD_VAL)
COL_ENE = COL_ENE or detectar(PAD_ENE)
assert COL_VAL in notas.columns and COL_ENE in notas.columns, (
    f"Este arquivo não tem as colunas de nota ({COL_VAL!r}, {COL_ENE!r}). "
    "Confira ARQ_NOTAS_MUSICAS na célula 1: talvez seja outro CSV.")

if "deezer_id" in notas.columns:
    notas["_chave"] = pd.to_numeric(notas.deezer_id, errors="coerce").astype("Int64").astype(str)
    indice["_chave"] = pd.to_numeric(indice.deezer_id, errors="coerce").astype("Int64").astype(str)
    ligacao = "deezer_id"
else:
    COL_ARQ = COL_ARQ or detectar(r"arquivo|file|path|caminho|^id$|nome")
    assert COL_ARQ, "Não achei coluna de ligação (deezer_id ou arquivo). Preencha COL_ARQ."
    chave = lambda s: s.astype(str).map(lambda x: Path(x).stem)
    notas["_chave"] = chave(notas[COL_ARQ]); indice["_chave"] = chave(indice["arquivo"])
    ligacao = COL_ARQ
notas = notas.drop_duplicates("_chave")
print(f"\nusando: valência={COL_VAL!r}  energia={COL_ENE!r}  ligação={ligacao!r}")

musicas = indice.merge(notas[["_chave", COL_VAL, COL_ENE]], on="_chave", how="inner")
musicas = musicas.rename(columns={COL_VAL: "val", COL_ENE: "ene"}).dropna(subset=["val", "ene"])
print(f"\n{len(notas)} linhas de notas | {len(indice)} prévias no índice | {len(musicas)} juntadas")
if len(musicas) < 0.95 * len(notas):
    print("ATENÇÃO: muitas notas não acharam par no índice; confira COL_ARQ")

## 3. Conferir as notas das músicas
### 3a. Distribuição
Duas perguntas: (1) as notas estão numa faixa razoável? (2) humor e energia saem muito correlacionados?
Se a correlação for alta (|r| > 0,6), os dois eixos dizem quase a mesma coisa, e a playlist vai diferenciar pouco "calma e feliz" de "agitada e feliz".

In [ ]:
import matplotlib.pyplot as plt

print(musicas[["val", "ene"]].describe().round(2).to_string())
print(f"\ncorrelação humor × energia nas músicas: {musicas.val.corr(musicas.ene):.2f}  "
      "(nas fotos do OASIS: -0,06)")

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(musicas.val, bins=50); ax[0].set_title("valência das músicas")
ax[1].hist(musicas.ene, bins=50); ax[1].set_title("energia das músicas")
ax[2].hexbin(musicas.val, musicas.ene, gridsize=40, mincnt=1, cmap="Blues")
ax[2].set(xlabel="valência", ylabel="energia", title="músicas no plano humor × energia")
plt.tight_layout(); plt.show()

### 3b. As notas batem com os rótulos de humor do MuSe?
Teste de sanidade barato: músicas marcadas como *energetic* deveriam ter mais energia que as marcadas como *calm*.
O **AUC** mede isso: 0,5 = a nota não separa os grupos; 1,0 = separa perfeitamente.

In [ ]:
from sklearn.metrics import roc_auc_score

musicas["lista_rotulos"] = musicas.rotulos.fillna("").str.split("|")
todos = set(r for rs in musicas.lista_rotulos for r in rs)

testes = {
    "ene": (["energetic", "rousing", "aggressive", "boisterous", "rowdy", "fiery", "intense", "exuberant"],
            ["calm", "peaceful", "soothing", "relaxed", "gentle", "serene", "tranquil", "laid-back/mellow"]),
    "val": (["cheerful", "happy", "joyous", "fun", "playful", "sweet", "warm", "euphoric"],
            ["sad", "melancholy", "gloomy", "bleak", "somber", "mournful", "bitter", "angst-ridden"]),
}
linhas = []
for eixo, (alto, baixo) in testes.items():
    alto = [r for r in alto if r in todos]; baixo = [r for r in baixo if r in todos]
    em = lambda rs, grupo: any(r in grupo for r in rs)
    a = musicas[musicas.lista_rotulos.map(lambda rs: em(rs, alto) and not em(rs, baixo))]
    b = musicas[musicas.lista_rotulos.map(lambda rs: em(rs, baixo) and not em(rs, alto))]
    if len(a) and len(b):
        y = np.r_[np.ones(len(a)), np.zeros(len(b))]
        auc = roc_auc_score(y, np.r_[a[eixo], b[eixo]])
        linhas.append({"eixo": "energia" if eixo == "ene" else "valência",
                       "alto (rótulos)": ", ".join(alto), "n alto": len(a),
                       "baixo (rótulos)": ", ".join(baixo), "n baixo": len(b),
                       "média alto": a[eixo].mean(), "média baixo": b[eixo].mean(), "AUC": auc})
pd.DataFrame(linhas).round(2)

### 3c. Limpeza: casos em que o MusiCNN contradiz os rótulos
O MusiCNN/emoMusic erra com metal extremo: dá valência fora da escala (< 1) e energia baixa.
Tiramos (1) valência < 1 e (2) músicas com rótulo agressivo e energia abaixo da mediana.
**Não rode a 3b de novo depois disto:** a limpeza usou os mesmos rótulos, a AUC subiria "de graça".

In [ ]:
pesados = [r for r in ["aggressive", "angry", "hostile", "menacing", "harsh", "brash", "volatile", "fiery"] if r in todos]
agressiva = musicas.lista_rotulos.map(lambda rs: any(r in rs for r in pesados))
fora_escala = musicas.val < 1
contraditoria = agressiva & (musicas.ene < musicas.ene.median())
remover = fora_escala | contraditoria
print(f"removendo {remover.sum()}: {fora_escala.sum()} fora da escala, {contraditoria.sum()} agressivas com energia baixa")
musicas = musicas[~remover].reset_index(drop=True)
print("restam", len(musicas))

### 3d. Popularidade
`pop` vai de 0 a 1: média do percentil da **faixa** (rank do Deezer) e do percentil do **artista** (fãs no Deezer).
Assim, uma faixa pouco tocada de um artista famoso (ex.: lado B dos Beatles) não é descartada.

In [ ]:
pop = pd.read_csv(ARQ_POP)
musicas = musicas.merge(pop[["deezer_id", "rank", "nb_fan", "pop", "genero"]], on="deezer_id", how="left")
musicas["genero"] = musicas["genero"].fillna("")
print(f"com popularidade: {musicas['pop'].notna().sum()} de {len(musicas)}")
antes = len(musicas)
musicas = musicas[musicas["pop"] >= POP_MIN].reset_index(drop=True)
print(f"POP_MIN={POP_MIN}: ficam {len(musicas)} de {antes}")
print("\nexemplos logo acima do corte (as menos conhecidas que ainda entram):")
print(musicas.nsmallest(10, "pop")[["artista", "titulo", "genero", "rank", "nb_fan", "pop"]].to_string())
print("\nmúsicas por gênero (opções que o usuário pode escolher na célula 6):")
print(musicas.genero.replace("", "(sem gênero)").value_counts().to_string())

## 4. Carregar o modelo de foto e o CLIP

In [ ]:
import joblib, torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

pac = joblib.load(ARQ_MODELO)
assert "media_prev" in pac, "Rode de novo a célula 6 da Parte 1 com as duas linhas novas"
print("modelo de", pac["data"], "| validação:", pac["validacao"])

disp = "cuda" if torch.cuda.is_available() else "cpu"
clip = CLIPModel.from_pretrained(pac["clip"]).eval().to(disp)
proc = CLIPProcessor.from_pretrained(pac["clip"])

@torch.no_grad()
def embeddings(imagens, lote=32):
    saidas = []
    for i in range(0, len(imagens), lote):
        x = proc(images=imagens[i:i + lote], return_tensors="pt").to(disp)
        e = clip.visual_projection(clip.vision_model(pixel_values=x["pixel_values"]).pooler_output)
        saidas.append(torch.nn.functional.normalize(e, dim=-1).cpu())
    return torch.cat(saidas).numpy()

## 5. A régua comum (z-score)
- **Foto:** `(previsão − media_prev) / desvio_prev`
- **Música:** `(nota − média das músicas) / desvio das músicas`

Depois disso, 0 = "típico" e ±1 = "um desvio acima/abaixo do típico", nos dois lados.

In [ ]:
m_mus = musicas[["val", "ene"]].mean().to_numpy()
d_mus = musicas[["val", "ene"]].std().to_numpy()
Zm = (musicas[["val", "ene"]].to_numpy() - m_mus) / d_mus

def z_foto(p):
    return (p - pac["media_prev"]) / pac["desvio_prev"]

print("média/desvio das músicas:", m_mus.round(2), d_mus.round(2))
print("média/desvio das previsões de foto:", pac["media_prev"].round(2), pac["desvio_prev"].round(2))

### Cobertura: as fotos caem onde há música?
Usamos as 900 fotos do OASIS como "fotos de exemplo". Para cada uma, medimos a distância até a 15ª música mais próxima.
Distância grande = a foto caiu numa região com pouca música, e a playlist vai sair "forçada".

In [ ]:
from sklearn.neighbors import NearestNeighbors

busca = NearestNeighbors(n_neighbors=TAMANHO_PLAYLIST).fit(Zm)

X_oasis = np.load(PASTA / "modelos" / "clip_oasis_embeddings.npy")
Zf = z_foto(pac["modelo"].predict(X_oasis))
dist, _ = busca.kneighbors(Zf)
d15 = dist[:, -1]
print(f"distância até a 15ª música: mediana {np.median(d15):.2f} | 95% das fotos até {np.quantile(d15, .95):.2f} | máx {d15.max():.2f}")

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.hexbin(Zm[:, 0], Zm[:, 1], gridsize=40, mincnt=1, cmap="Greys", alpha=.8)
sc = ax.scatter(Zf[:, 0], Zf[:, 1], c=d15, s=12, cmap="viridis_r")
plt.colorbar(sc, label="distância até a 15ª música")
ax.set(xlabel="valência (z)", ylabel="energia (z)", title="cinza: músicas | pontos: fotos do OASIS")
plt.tight_layout(); plt.show()

## 6. Gerar a playlist de uma foto
Busca as músicas mais próximas no plano (z) e respeita `MAX_POR_ARTISTA`.
Se a prévia estiver no Drive, dá para ouvir ali mesmo.

In [ ]:
from IPython.display import Audio, Markdown, display
from google.colab import files
import io

# Gêneros escolhidos pelo usuário (vazio = todos). Use os nomes da lista da célula 3d.
# Ex.: GENEROS = ["Rock", "Alternative"]. Para trocar, mude aqui e rode só esta célula.
GENEROS = []

def playlist(imagem, generos=GENEROS, n=TAMANHO_PLAYLIST, max_por_artista=MAX_POR_ARTISTA):
    p = pac["modelo"].predict(embeddings([imagem]))[0]
    z = z_foto(p)
    # filtra pelos gêneros escolhidos; a régua (z) continua a mesma para todos
    pos = np.flatnonzero(musicas.genero.isin(generos)) if generos else np.arange(len(musicas))
    if len(pos) < n:
        raise ValueError(f"Só {len(pos)} músicas nos gêneros {generos}; escolha mais gêneros.")
    dist, idx = NearestNeighbors(n_neighbors=min(len(pos), n * CANDIDATOS_POR_VAGA)).fit(Zm[pos]).kneighbors([z])
    idx = pos[idx]
    # mistura proximidade e popularidade (menor nota = melhor)
    ordem = sorted(zip(dist[0], idx[0]), key=lambda t: t[0] - ALFA * musicas["pop"].iloc[t[1]])
    escolhidas, por_artista = [], {}
    for d, i in ordem:
        art = str(musicas.artista.iloc[i]).lower()
        if por_artista.get(art, 0) >= max_por_artista:
            continue
        por_artista[art] = por_artista.get(art, 0) + 1
        escolhidas.append((i, d))
        if len(escolhidas) == n:
            break
    tab = musicas.iloc[[i for i, _ in escolhidas]][["artista", "titulo", "genero", "rotulos", "val", "ene", "pop", "deezer_id", "arquivo"]].copy()
    tab["distância"] = [d for _, d in escolhidas]
    tab["deezer"] = "https://www.deezer.com/track/" + tab.deezer_id.astype("Int64").astype(str)
    return p, z, tab.reset_index(drop=True)

enviadas = files.upload()
for nome, conteudo in enviadas.items():
    img = Image.open(io.BytesIO(conteudo)).convert("RGB")
    p, z, tab = playlist(img)
    display(img.resize((300, int(300 * img.height / img.width))))
    display(Markdown(f"**{nome}** → valência {p[0]:.2f}, energia {p[1]:.2f} (escala OASIS)  |  "
                     f"z = ({z[0]:+.2f}, {z[1]:+.2f})  |  distância média {tab['distância'].mean():.2f}"
                     f"  |  gêneros: {', '.join(GENEROS) or 'todos'}"))
    if tab["distância"].mean() > 0.5:
        display(Markdown("⚠️ *Poucas músicas desses gêneros perto do clima da foto: a playlist pode sair forçada.*"))
    display(tab.drop(columns=["arquivo", "deezer_id"]).round(2))

## 6b. Playlist final, com **modo festa**
O modelo de foto (OASIS) acha foto de festa "calma": no OASIS, *Party*, *Drink*, *Alcohol* têm arousal ~2–3.
Correção provisória (regra manual, não aprendida):
1. O CLIP classifica o tipo de cena (festa/balada/comemoração × paisagem, praia, retrato, cidade…).
2. Se `prob_festa ≥ LIMIAR_FESTA`: o alvo da foto é puxado para **alegre + agitada**, e só entram músicas de
   `GEN_FESTA` (ou dos gêneros escolhidos pelo usuário) com energia e humor acima da média.
3. Senão, nada muda.

Próximo passo para trocar a regra por algo aprendido: classificador `danceability` da Essentia nas prévias.

In [ ]:
CENAS_FESTA = ["a photo of a party", "a photo of people dancing at a nightclub",
               "a photo of a celebration with friends", "a photo of a concert crowd"]
CENAS_OUTRAS = ["a photo of a landscape", "a photo of a beach", "a portrait of a person",
                "a photo of a city street", "a photo of nature", "a photo of food",
                "a photo of a room", "a selfie", "a photo of animals"]
GEN_FESTA = ["Pop", "Dance", "Electro", "Rap/Hip Hop", "R&B"]
LIMIAR_FESTA = 0.5
ALVO_FESTA = (0.5, 1.0)
BETA_FESTA = 0.5             # peso dos rótulos do CLIP para diferenciar festas entre si (testado: 4/15 em comum)
TOP_ROTULOS = 15

@torch.no_grad()
def emb_texto(frases):
    x = proc(text=frases, return_tensors="pt", padding=True).to(disp)
    t = clip.text_projection(clip.text_model(input_ids=x["input_ids"], attention_mask=x["attention_mask"]).pooler_output)
    return torch.nn.functional.normalize(t, dim=-1).cpu().numpy()

T_cenas = emb_texto(CENAS_FESTA + CENAS_OUTRAS)
rot = sorted({r for rs in musicas.lista_rotulos for r in rs if r})
T_rot = emb_texto([f"a photo with a {r} mood" for r in rot])
idx_rot = {r: i for i, r in enumerate(rot)}
listas = musicas.lista_rotulos.map(lambda rs: [idx_rot[r] for r in rs if r in idx_rot])

def prob_festa(e):
    s = 100 * (T_cenas @ e); p = np.exp(s - s.max()); p /= p.sum()
    return float(p[:len(CENAS_FESTA)].sum())

def nota_rotulos(e):
    s = T_rot @ e; top = np.argsort(-s)[:TOP_ROTULOS]
    w = np.zeros(len(rot)); ex = np.exp(100 * (s[top] - s[top].max())); w[top] = ex / ex.max()
    return listas.map(lambda ls: max((w[i] for i in ls), default=0.0)).to_numpy(), [rot[i] for i in top[:6]]

from collections import Counter
CANDIDATOS_GENERO = 30
GEN_DEEZER_SITE = {"Pop": "Pop", "Rock": "Rock", "Alternativo": "Alternativo", "R&B": "R&B", "Rap/Hip Hop": "Rap/Hip Hop",
                   "Electro": "Eletrônica", "Dance": "Eletrônica", "Filmes/Games": "Trilhas sonoras"}
VIZINHOS_SITE = {"Pop": ["R&B", "Alternativo"], "Rock": ["Alternativo", "Pop"], "Alternativo": ["Rock", "Pop"],
                 "R&B": ["Pop", "Rap/Hip Hop"], "Rap/Hip Hop": ["R&B", "Pop"], "Eletrônica": ["Pop", "Alternativo"],
                 "Trilhas sonoras": ["Alternativo", "Pop"]}
GEN_SITE = musicas.genero.map(GEN_DEEZER_SITE).fillna("").to_numpy()
POP_ARR = musicas["pop"].to_numpy()
# artista-base ("Bob Marley & The Wailers" = "Bob Marley") e deezer_id, para não repetir música
ART_BASE = musicas.artista.astype(str).str.lower().str.split(r" & | feat\.? | ft\.? |, | and ", regex=True).str[0].str.strip().to_numpy()
DZ = musicas.deezer_id.to_numpy()

def nota_perto(dist, ok, k):
    k = min(int(ok.sum()), k)
    perto = np.argsort(np.where(ok, dist, np.inf))[:k]
    nota = np.full(len(musicas), np.inf)
    nota[perto] = dist[perto] - ALFA * POP_ARR[perto]
    return nota

def escolher(nota, n, max_por_artista):
    esc, por_artista, ids = [], {}, set()
    for i in np.argsort(nota):
        if not np.isfinite(nota[i]): break
        a = ART_BASE[i]
        if por_artista.get(a, 0) >= max_por_artista or DZ[i] in ids: continue
        por_artista[a] = por_artista.get(a, 0) + 1; ids.add(DZ[i]); esc.append(i)
        if len(esc) == n: break
    return esc

def playlist_final(img, generos=GENEROS, n=TAMANHO_PLAYLIST, max_por_artista=MAX_POR_ARTISTA):
    e = embeddings([img])[0]
    p = pac["modelo"].predict(e[None])[0]; z = z_foto(p)
    pf = prob_festa(e); festa = pf >= LIMIAR_FESTA
    gen_ok = musicas.genero.isin(generos).to_numpy() if generos else np.ones(len(musicas), bool)
    tag, rots = nota_rotulos(e)
    if festa:
        alvo = np.array([max(z[0], ALVO_FESTA[0]), max(z[1], ALVO_FESTA[1])])
        if not generos:
            gen_ok = musicas.genero.isin(GEN_FESTA).to_numpy()
        ok = gen_ok & (Zm[:, 0] > 0) & (Zm[:, 1] > 0.3)
        if ok.sum() < n * 2:
            ok = gen_ok
        bonus = BETA_FESTA * tag
        dist = np.linalg.norm(Zm - alvo, axis=1)
        nota = np.where(ok, dist - ALFA * POP_ARR - bonus, np.inf)
    else:
        # modo normal = igual ao site: só as 90 mais próximas são reordenadas pela popularidade
        dist = np.linalg.norm(Zm - z, axis=1)
        nota = nota_perto(dist, gen_ok, n * CANDIDATOS_POR_VAGA)
        if not generos:
            # sem gênero escolhido: gênero mais comum da playlist provisória + 1 vizinho (15/11)
            cont = Counter(g for g in GEN_SITE[escolher(nota, n, max_por_artista)] if g in VIZINHOS_SITE)
            if cont:
                dom = cont.most_common(1)[0][0]
                nota = nota_perto(dist, np.isin(GEN_SITE, [dom, VIZINHOS_SITE[dom][0]]), CANDIDATOS_GENERO)
    escolhidas = escolher(nota, n, max_por_artista)
    tab = musicas.iloc[escolhidas][["artista", "titulo", "genero", "rotulos", "val", "ene", "pop", "deezer_id", "arquivo"]].copy()
    tab["rótulo_match"] = tag[escolhidas]
    tab["deezer"] = "https://www.deezer.com/track/" + tab.deezer_id.astype("Int64").astype(str)
    return tab.drop(columns="deezer_id").reset_index(drop=True), p, pf, festa, rots

resultados = {}
for nome, conteudo in files.upload().items():
    img = Image.open(io.BytesIO(conteudo)).convert("RGB")
    tab, p, pf, festa, rots = playlist_final(img)
    resultados[nome] = set(tab.artista + " — " + tab.titulo)
    display(img.resize((250, int(250 * img.height / img.width))))
    display(Markdown(f"**{nome}** → prob. festa **{pf:.2f}** → modo festa **{'SIM' if festa else 'não'}** | rótulos: {', '.join(rots)}"))
    display(tab.drop(columns="arquivo").round(2))

nomes = list(resultados)
for a in range(len(nomes)):
    for b in range(a + 1, len(nomes)):
        print(f"em comum entre {nomes[a]} e {nomes[b]}: {len(resultados[nomes[a]] & resultados[nomes[b]])} de {TAMANHO_PLAYLIST}")

### Ouvir as 5 primeiras (se as prévias estiverem no Drive)

In [ ]:
for _, linha in tab.head(5).iterrows():
    caminho = Path("/content/drive/MyDrive/playlist-foto") / linha.arquivo
    print(f"{linha.artista} — {linha.titulo}")
    if caminho.exists():
        display(Audio(str(caminho)))
    else:
        print("   (prévia não encontrada em", caminho, ")")